In [1]:
import pandas as pd

df = pd.read_csv("supply_chain_demand_data.csv")
print(df.head())

   sales_date  store_id  product_id              product_name  supplier_id  \
0  2024-01-01         1         101  Electronics - Smartphone            1   
1  2024-01-01         2         101  Electronics - Smartphone            1   
2  2024-01-01         3         101  Electronics - Smartphone            1   
3  2024-01-01         1         102      Electronics - Laptop            1   
4  2024-01-01         2         102      Electronics - Laptop            1   

          supplier_name country  cost_price  unit_price  on_promotion  \
0  Global Logistics Co.     USA       300.0       500.0             1   
1  Global Logistics Co.     USA       300.0       500.0             0   
2  Global Logistics Co.     USA       300.0       500.0             0   
3  Global Logistics Co.     USA       700.0      1100.0             1   
4  Global Logistics Co.     USA       700.0      1100.0             0   

   units_sold  
0         106  
1          85  
2          48  
3         107  
4          5

In [2]:
import pandas as pd

df = pd.read_csv("supply_chain_demand_data.csv")
df["sales_date"] = pd.to_datetime(df["sales_date"])

df = df.sort_values(by=["store_id", "product_id", "sales_date"])

df["lag_1_sales"] = df.groupby(["store_id", "product_id"])["units_sold"].shift(1)
df["rolling_7_avg"] = df.groupby(["store_id", "product_id"])["units_sold"].transform(lambda x: x.shift(1).rolling(window=7, min_periods=1).mean())

df = df.fillna(0)

print(df[["sales_date", "store_id", "product_id", "units_sold", "lag_1_sales", "rolling_7_avg"]].head(5))

   sales_date  store_id  product_id  units_sold  lag_1_sales  rolling_7_avg
0  2024-01-01         1         101         106          0.0       0.000000
15 2024-01-02         1         101          31        106.0     106.000000
30 2024-01-03         1         101          86         31.0      68.500000
45 2024-01-04         1         101          59         86.0      74.333333
60 2024-01-05         1         101          37         59.0      70.500000


In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

sub_df = df[(df['store_id'] == 1) & (df['product_id'] == 101)].sort_values('sales_date').copy()

features = ['units_sold', 'lag_1_sales', 'rolling_7_avg', 'on_promotion']
data = sub_df[features].values

scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

def create_sequences(dataset, time_step=30):
    X, y = [], []
    for i in range(time_step, len(dataset)):
        X.append(dataset[i-time_step:i, :])
        y.append(dataset[i, 0])
    return np.array(X), np.array(y)

time_step = 30
X, y = create_sequences(scaled_data, time_step)

train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    LSTM(50, return_sequences=False),
    Dropout(0.2),
    Dense(25),
    Dense(1)
])

model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(X_train, y_train, batch_size=32, epochs=10, validation_data=(X_test, y_test))

print("LSTM Model Training Completed Successfully!")

ModuleNotFoundError: No module named 'tensorflow'